# Interim data prep: CREMA-D and RAVDESS

Goal: turn the two raw downloaded datasets into the same shape the existing Stage-3 training code already consumes. No pipeline code needs changing downstream.

End state after running this notebook:

```
data/crema_d/
    VideoFlash/                  <- raw .flv (or .mp4) from Cheyney release
    AudioWAV/                    <- raw .wav
    cropped_aligned/<clip>/<00000..>.jpg      <- MediaPipe crops
    wav/<clip>.wav                            <- ffmpeg 16k mono
    annotations/{train,val,test}.txt          <- AffWild2-shaped

data/ravdess/                same sub-layout (speech clips only)

cache/features/crema_d/enet_b0_8_va_mtl/<clip>.npz
cache/features/crema_d/hubert_large/<clip>.npz
cache/features/crema_d/hubert_large_aligned/<clip>.npz
    ...and the mirror for ravdess.
```

**Prereqs.** `ffmpeg` on PATH; `pip install mediapipe opencv-python` (already in `pyproject.toml` optional); the raw CREMA-D and RAVDESS archives already expanded under `data/`.

Sections:
1. Imports + path setup
2. Parse clips from filenames
3. Face crops (MediaPipe)
4. Audio WAVs (ffmpeg)
5. Visual features (EmotiEffLib)
6. Audio features (HuBERT)
7. Align audio to visual timeline
8. Write AffWild2-shaped annotations with actor-disjoint splits
9. Sanity checks

## 1. Imports + path setup

In [5]:
2+1

3

In [6]:
from __future__ import annotations

import sys
from pathlib import Path

# Allow notebook-from-notebooks/ to import 'src.*'.
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)

DATA = ROOT / "data"
CACHE = ROOT / "cache"

CREMA = {
    "video":       DATA / "crema_d" / "VideoFlash",
    "audio_src":   DATA / "crema_d" / "AudioWAV",     # original 48k stereo
    "crops":       DATA / "crema_d" / "cropped_aligned",
    "wav16":       DATA / "crema_d" / "wav",           # 16k mono for HuBERT
    "annotations": DATA / "crema_d" / "annotations",
    "visual_npz":  CACHE / "features" / "crema_d" / "enet_b0_8_va_mtl",
    "visual_npz_mbf": CACHE / "features" / "crema_d" / "mbf_va_mtl",
    "audio_npz":   CACHE / "features" / "crema_d" / "hubert_large",
    "audio_al":    CACHE / "features" / "crema_d" / "hubert_large_aligned",
}

RAVDESS = {
    "video":       DATA / "ravdess" / "Video_Speech",   # Actor_01/... subdirs
    "crops":       DATA / "ravdess" / "cropped_aligned",
    "wav16":       DATA / "ravdess" / "wav",
    "annotations": DATA / "ravdess" / "annotations",
    "visual_npz":  CACHE / "features" / "ravdess" / "enet_b0_8_va_mtl",
    "audio_npz":   CACHE / "features" / "ravdess" / "hubert_large",
    "audio_al":    CACHE / "features" / "ravdess" / "hubert_large_aligned",
}

for d in list(CREMA.values()) + list(RAVDESS.values()):
    d.mkdir(parents=True, exist_ok=True)

project root: C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code


## 2. Parse clips from filenames

`parse_crema_d_clips` reads `1001_DFA_ANG_XX.flv` style stems; `parse_ravdess_clips` reads the 7-field RAVDESS stems and excludes song + audio-only modalities by default.

In [7]:
from src.datasets.interim_prep import (
    parse_crema_d_clips, parse_ravdess_clips, actor_split, split_summary,
)

crema_clips = parse_crema_d_clips(
    video_dir=CREMA["video"],
    audio_dir=CREMA["audio_src"],
    video_ext=".flv",     # change to '.mp4' if using the transcoded mirror
)
print(f"CREMA-D: {len(crema_clips)} clips; {len({c.actor_id for c in crema_clips})} actors")

ravdess_clips = parse_ravdess_clips(
    video_dir=RAVDESS["video"], include_song=False, video_ext=".mp4",
)
print(f"RAVDESS speech: {len(ravdess_clips)} clips; {len({c.actor_id for c in ravdess_clips})} actors")

CREMA-D: 7442 clips; 91 actors
RAVDESS speech: 2880 clips; 24 actors


## 3. Face crops

One MediaPipe pass over every clip, target 5 fps, 112x112 crops written to `cropped_aligned/<clip>/<idx:05d>.jpg`. Videos with no detectable face (expected to be rare on studio data) yield an empty folder and are dropped at annotation-write time.

In [4]:
from src.features.extract_faces_from_video import (
    FaceCropConfig, extract_faces_for_clips,
)

cfg = FaceCropConfig(output_size=112, target_fps=5.0, expand_ratio=0.30)

crema_counts = extract_faces_for_clips(
    video_paths=[c.source_video_path for c in crema_clips],
    output_dir=CREMA["crops"],
    cfg=cfg,
    skip_existing=True,
)
print(f"CREMA-D: extracted faces for {sum(1 for v in crema_counts.values() if v > 0)}/{len(crema_counts)} clips")

ravdess_counts = extract_faces_for_clips(
    video_paths=[c.source_video_path for c in ravdess_clips],
    output_dir=RAVDESS["crops"],
    cfg=cfg,
    skip_existing=True,
)
print(f"RAVDESS: extracted faces for {sum(1 for v in ravdess_counts.values() if v > 0)}/{len(ravdess_counts)} clips")

faces:   0%|          | 0/7442 [00:00<?, ?it/s]

faces: 100%|██████████| 7442/7442 [19:09<00:00,  6.48it/s]


CREMA-D: extracted faces for 7441/7442 clips


faces: 100%|██████████| 2880/2880 [24:05<00:00,  1.99it/s]

RAVDESS: extracted faces for 2880/2880 clips


## 4. Audio WAVs (16 kHz mono)

CREMA-D's stock WAVs are 48k stereo; HuBERT wants 16k mono. RAVDESS has no separate WAV archive, so we pull audio out of the video. We use the existing `extract_audio_wavs` helper which shells out to ffmpeg.

In [2]:
from src.features.extract_audio_wav import extract_audio_wavs

# CREMA-D: the videos carry no audio track -> transcode the shipped WAVs instead.
# Put them beside the .flv so extract_audio_wavs() picks them up from one dir.
# If your mirror already ships mp4 with audio, point this at CREMA['video'] directly.
extract_audio_wavs(
    videos_dir=CREMA["audio_src"],    # accepts .wav too via the default codec path
    output_dir=CREMA["wav16"],
    sample_rate=16000, channels=1, overwrite=False,
)

extract_audio_wavs(
    videos_dir=RAVDESS["video"],
    output_dir=RAVDESS["wav16"],
    sample_rate=16000, channels=1, overwrite=False,
)

ffmpeg: 100%|██████████| 7442/7442 [07:03<00:00, 17.59it/s]


[audio] wrote 7442 wav files to C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\data\crema_d\wav


ffmpeg: 100%|██████████| 2880/2880 [03:43<00:00, 12.90it/s]

[audio] wrote 2880 wav files to C:\Users\Andrey Lyaschenko\Documents\vkr\thesis-code\data\ravdess\wav


[WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/data/ravdess/wav/01-01-01-01-01-01-01.wav'),
 WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/data/ravdess/wav/01-01-01-01-01-02-01.wav'),
 WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/data/ravdess/wav/01-01-01-01-02-01-01.wav'),
 WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/data/ravdess/wav/01-01-01-01-02-02-01.wav'),
 WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/data/ravdess/wav/01-01-02-01-01-01-01.wav'),
 WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/data/ravdess/wav/01-01-02-01-01-02-01.wav'),
 WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/data/ravdess/wav/01-01-02-01-02-01-01.wav'),
 WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/data/ravdess/wav/01-01-02-01-02-02-01.wav'),
 WindowsPath('C:/Users/Andrey Lyaschenko/Documents/vkr/thesis-code/data/ravdess/wav/01-01-02-02-01-01-01

## 5. Visual features (EmotiEffLib)

Runs `enet_b0_8_va_mtl` over every `<clip>/*.jpg` folder and writes `<clip>.npz` with `features (N,1280)`, `scores (N,10)`, `image_names (N,)`. Also does the MBF variant so we can run the backbone ablation in notebook 03 without re-cropping.

In [4]:
from src.features.extract_visual import extract_visual_features

extract_visual_features(
    model_name="enet_b0_8_va_mtl",
    cropped_aligned_dir=CREMA["crops"],
    output_dir=CREMA["visual_npz"],
    batch_size=48,
)

extract_visual_features(
    model_name="mbf_va_mtl",
    cropped_aligned_dir=CREMA["crops"],
    output_dir=CREMA["visual_npz_mbf"],
    batch_size=48,
)

extract_visual_features(
    model_name="enet_b0_8_va_mtl",
    cropped_aligned_dir=RAVDESS["crops"],
    output_dir=RAVDESS["visual_npz"],
    batch_size=48,
)

extract[enet_b0_8_va_mtl]: 100%|██████████| 2880/2880 [14:28<00:00,  3.32it/s]


## 6. Audio features (HuBERT large)

Frozen HuBERT at ~50 Hz, one `.npz` per clip with `features (T_audio, 1024)`, `hop_sec`, `wav_seconds`.

In [3]:
from src.features.extract_audio_features import extract_audio_features

extract_audio_features(
    backbone="hubert_large",
    wav_dir=CREMA["wav16"],
    output_dir=CREMA["audio_npz"],
)

extract_audio_features(
    backbone="hubert_large",
    wav_dir=RAVDESS["wav16"],
    output_dir=RAVDESS["audio_npz"],
)

c:\Users\Andrey Lyaschenko\Documents\vkr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 422/422 [00:00<00:00, 16286.01it/s]
HubertModel LOAD REPORT from: facebook/hubert-large-ls960-ft
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 422/422 [00:00<00:00, 67588.07it/s]
HubertModel LOAD REPORT from: facebook/hubert-large-ls960-ft
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archi

## 7. Align audio to visual timeline

Linear interpolation onto the same frame count as the visual cache; emits `features (T_video, D_a)`, `has_audio`.

In [3]:
from src.features.align_audio_to_video import align_caches

align_caches(
    audio_cache_dir=CREMA["audio_npz"],
    visual_cache_dir=CREMA["visual_npz"],
    output_dir=CREMA["audio_al"],
    fallback_dim=1024,
)

align_caches(
    audio_cache_dir=RAVDESS["audio_npz"],
    visual_cache_dir=RAVDESS["visual_npz"],
    output_dir=RAVDESS["audio_al"],
    fallback_dim=1024,
)

## 8. Write AffWild2-shaped annotations

Drop any clip whose face-extraction produced zero frames (MediaPipe miss) -- the visual cache has nothing to train on.

Split is 70/15/15 by actor; CREMA-D has 91 speakers and RAVDESS 24, so the splits are well-defined.

In [8]:
import numpy as np
from src.datasets.interim_prep import write_annotations

def frame_count_from_visual_cache(npz_dir: Path):
    """Return a callable that gives len(features) for a clip record."""
    cache = {
        p.stem: int(np.load(p, allow_pickle=True)["features"].shape[0])
        for p in npz_dir.glob("*.npz")
    }
    return lambda clip: cache.get(clip.videoname, 0)


# CREMA-D -- drop zero-frame clips, 70/15/15 actor split.
crema_fc = frame_count_from_visual_cache(CREMA["visual_npz"])
crema_usable = [c for c in crema_clips if crema_fc(c) > 0]
crema_train, crema_val, crema_test = actor_split(
    crema_usable, val_frac=0.15, test_frac=0.15, seed=42,
)
for split_name, split_clips in (("train", crema_train), ("val", crema_val), ("test", crema_test)):
    n = write_annotations(
        split_clips,
        output_file=CREMA["annotations"] / f"{split_name}.txt",
        frame_count_for=crema_fc,
    )
    print(f"CREMA-D {split_name}: {len(split_clips)} clips, {n} rows")

print("CREMA-D class balance:", split_summary(crema_train, crema_val, crema_test))

CREMA-D train: 5152 clips, 67701 rows
CREMA-D val: 1147 clips, 14988 rows
CREMA-D test: 1142 clips, 15302 rows
CREMA-D class balance: {'train': {'Anger': 880, 'Disgust': 880, 'Fear': 880, 'Happiness': 880, 'Neutral': 753, 'Sadness': 879}, 'val': {'Anger': 196, 'Disgust': 196, 'Fear': 196, 'Happiness': 196, 'Neutral': 167, 'Sadness': 196}, 'test': {'Anger': 195, 'Disgust': 195, 'Fear': 195, 'Happiness': 195, 'Neutral': 167, 'Sadness': 195}}


In [9]:
# RAVDESS -- same thing. Smaller, so widen the val/test to 20/20.
ravdess_fc = frame_count_from_visual_cache(RAVDESS["visual_npz"])
ravdess_usable = [c for c in ravdess_clips if ravdess_fc(c) > 0]
r_train, r_val, r_test = actor_split(
    ravdess_usable, val_frac=0.20, test_frac=0.20, seed=42,
)
for split_name, split_clips in (("train", r_train), ("val", r_val), ("test", r_test)):
    n = write_annotations(
        split_clips,
        output_file=RAVDESS["annotations"] / f"{split_name}.txt",
        frame_count_for=ravdess_fc,
    )
    print(f"RAVDESS {split_name}: {len(split_clips)} clips, {n} rows")
print("RAVDESS class balance:", split_summary(r_train, r_val, r_test))

RAVDESS train: 1680 clips, 31645 rows
RAVDESS val: 600 clips, 11588 rows
RAVDESS test: 600 clips, 11050 rows
RAVDESS class balance: {'train': {'Anger': 224, 'Calm': 224, 'Disgust': 224, 'Fear': 224, 'Happiness': 224, 'Neutral': 112, 'Sadness': 224, 'Surprise': 224}, 'val': {'Anger': 80, 'Calm': 80, 'Disgust': 80, 'Fear': 80, 'Happiness': 80, 'Neutral': 40, 'Sadness': 80, 'Surprise': 80}, 'test': {'Anger': 80, 'Calm': 80, 'Disgust': 80, 'Fear': 80, 'Happiness': 80, 'Neutral': 40, 'Sadness': 80, 'Surprise': 80}}


## 9. Sanity checks

Round-trip the written annotation file through the real `read_mtl_annotations` parser: this is the same function training uses, so if this line doesn't raise, training won't either.

In [12]:
from src.datasets.affwild2_mtl import read_mtl_annotations

# Build the features_index the parser expects: the exact set of image paths
# we put in the annotation file. Matches the cropped_aligned layout.
def features_index(clips, frame_count):
    idx = set()
    for c in clips:
        T = frame_count(c)
        for fi in range(T):
            idx.add(f"{c.videoname}/{fi:05d}.jpg")
    return idx

anno = read_mtl_annotations(
    CREMA["annotations"] / "train.txt",
    features_index=features_index(crema_train, crema_fc),
)
print("CREMA-D train rows loaded:", len(anno))
print("  mask_va on :", int((anno.mask_va == 1).sum()))
print("  mask_expr on:", int((anno.mask_expr == 1).sum()))
print("  mask_au  on:", int((anno.mask_au == 1).sum()), "(should be 0 -- AU disabled)")
print(anno.df.head(3))

CREMA-D train rows loaded: 67701
  mask_va on : 67701
  mask_expr on: 67701
  mask_au  on: 0 (should be 0 -- AU disabled)
                  image_path        videoname  frame_index  valence  arousal  \
0  1001_DFA_ANG_XX/00000.jpg  1001_DFA_ANG_XX            0     -0.7      0.7   
1  1001_DFA_ANG_XX/00001.jpg  1001_DFA_ANG_XX            1     -0.7      0.7   
2  1001_DFA_ANG_XX/00002.jpg  1001_DFA_ANG_XX            2     -0.7      0.7   

   expr  mask_va  mask_expr  mask_au  au1  ...  au3  au4  au5  au6  au7  au8  \
0     1      1.0        1.0      0.0    0  ...    0    0    0    0    0    0   
1     1      1.0        1.0      0.0    0  ...    0    0    0    0    0    0   
2     1      1.0        1.0      0.0    0  ...    0    0    0    0    0    0   

   au9  au10  au11  au12  
0    0     0     0     0  
1    0     0     0     0  
2    0     0     0     0  

[3 rows x 21 columns]
